# 02 p3 development diagnostics

**Permitted partitions: TRAIN ONLY**

Inspect training-only diagnostics without revising the frozen protocol. Real runs fetch the verified archive temporarily; only development rows are exposed.

Protocol: frozen 2026-10-08; provenance uses exact content hashes. Training targets end 2015-02-28; validation ends 2016-02-29. Final test outcomes are not analyzed here.

Prerequisites: open this notebook from the same P4 Git revision you will enter below. In Colab Secrets, set `AIRSENSE_GITHUB_TOKEN` with write access to this repository. Outputs go to `results/<run-id>` on a dedicated Git branch after every checkpoint. Expect setup/download to take minutes; no forecasting model training occurs. If dependency installation requires a runtime restart, restart and rerun setup.


## Runtime bootstrap
Clone a pinned revision in a fresh runtime; preserve existing work on rerun.

In [1]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = "https://github.com/Engr-Daniel/airsense-r.git"
# Use the full commit SHA of the GitHub revision containing this notebook.
# Resume using the SAME revision and run ID as the original execution.
TEST_MODE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_MODE") == "1"
if TEST_MODE:
    ROOT = Path(os.environ["AIRSENSE_NOTEBOOK_TEST_ROOT"]).resolve()
else:
    CODE_REVISION = (os.environ.get("AIRSENSE_CODE_REVISION") or input("Code version: full 40-character Git commit SHA (not a run ID): ")).strip().lower()
    if len(CODE_REVISION) != 40 or any(c not in "0123456789abcdef" for c in CODE_REVISION.lower()):
        raise ValueError("Use a full Git commit SHA, not a moving branch name")
    ROOT = Path.cwd() / "airsense-r"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", CODE_REVISION], check=True)
    else:
        remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
        head = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
        if remote != REPOSITORY or head != CODE_REVISION or dirty:
            raise RuntimeError("Existing checkout differs or has edits; preserve it and use a fresh runtime")
    if sys.version_info < (3, 11):
        raise RuntimeError("Python 3.11+ required")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-p4.txt"), "-e", str(ROOT)], check=True)
    print("Code revision:", CODE_REVISION)
# Editable installs may not activate their .pth file in an already-running kernel.
# Activate only this checkout, and reject cached modules from another checkout.
import importlib
SOURCE_DIR = (ROOT / "src").resolve()
PACKAGE_DIR = SOURCE_DIR / "airsense_r"
if not (PACKAGE_DIR / "__init__.py").is_file():
    raise RuntimeError("Selected checkout has no airsense_r package")
for module_name, module in tuple(sys.modules.items()):
    if module_name == "airsense_r" or module_name.startswith("airsense_r."):
        location = getattr(module, "__file__", None)
        if location is None or not Path(location).resolve().is_relative_to(PACKAGE_DIR):
            raise RuntimeError("Another airsense_r checkout is already imported; restart the runtime")
source_path = str(SOURCE_DIR)
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)
importlib.invalidate_caches()
import airsense_r
if Path(airsense_r.__file__).resolve() != PACKAGE_DIR / "__init__.py":
    raise RuntimeError("Imported package does not match the selected checkout")
print("Workspace:", ROOT)
print("Verified package:", airsense_r.__file__)


Code version: full 40-character Git commit SHA (not a run ID): fd612fc6ed3a7eaa6b074bd30b406f7725cf4e17
Code revision: fd612fc6ed3a7eaa6b074bd30b406f7725cf4e17
Workspace: /content/airsense-r
Verified package: /content/airsense-r/src/airsense_r/__init__.py


## Artifact authentication and run identity
The secret is never printed or written into Git configuration. The code SHA selects the software version; the run ID names this notebook's outputs. Use a different run ID for each notebook. Each run pushes compact artifacts to its own `results/<run-id>` branch, keeping source on `main` separate. Reuse the same run ID and original code/environment only for compatible resumes. Updated code requires a new run ID; historical results remain valid under their original revision.

In [2]:
import logging
from airsense_r.artifacts import provenance, json_bytes
from airsense_r.artifacts import source_hash
from airsense_r.checkpoints import open_run
from airsense_r.colab import configure_github_secret

logging.basicConfig(level=logging.INFO)
if source_hash(ROOT) != "14e9f1491af8a5b10c82e0bc1f137ba909da437880a809db3cc62a9f60fda90a":
    raise RuntimeError("Notebook and scientific source revisions differ; open the matching notebook revision")
if TEST_MODE:
    RUN_ID = os.environ["AIRSENSE_NOTEBOOK_TEST_RUN"]
    REMOTE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_REMOTE")
else:
    from google.colab import userdata
    configure_github_secret(ROOT, userdata.get("AIRSENSE_GITHUB_TOKEN"))
    RUN_ID = input("Run name, e.g. nb03-validation-01 (same name only for resume; not the code SHA): ").strip()
    REMOTE = REPOSITORY


Run name, e.g. nb03-validation-01 (same name only for resume; not the code SHA): nb02-diagnostics-02


## Evidence and validation
Provenance is checked before restoring outputs. Each completed artifact is saved and pushed before continuing.

In [3]:
from airsense_r.p4 import run_pipeline, synthetic_stream
if TEST_MODE:
    store = open_run(ROOT, RUN_ID, provenance(ROOT, {"notebook": "02", "synthetic_only": True}), REMOTE)
    data = synthetic_stream()
    summary = data.groupby("station")["PM2.5"].agg(["count", "mean"])
    store.save("synthetic-diagnostics.csv", summary.to_csv().encode())
    display(summary)
else:
    output = run_pipeline(ROOT, RUN_ID, remote=REMOTE, training_diagnostics=True)
    print("Verified training summaries saved incrementally:", output)
    from airsense_r.p4 import inspect_training_diagnostics
    display(inspect_training_diagnostics(ROOT, output))


Verified training summaries saved incrementally: /content/airsense-r/.tmp/checkpoint-checkouts/nb02-diagnostics-02/results/nb02-diagnostics-02


,station,fit_end,CO,DEWP,NO2,O3,PM10,PM2.5,PRES,RAIN,SO2,TEMP,WSPM
0,Aotizhongxin,2015-02-28 23:00,1237.636533,3.685729,63.714516,48.057842,117.866767,86.998623,1010.738361,0.058728,23.404255,13.454356,1.521949
1,Changping,2015-02-28 23:00,1181.608230,1.231662,44.461516,59.930126,100.825111,77.546601,1007.482422,0.046987,21.322090,13.833995,1.787591
2,Dingling,2015-02-28 23:00,904.782048,1.231662,29.274435,66.966646,86.227262,70.966154,1007.482422,0.046987,15.283074,13.833995,1.787591
3,Dongsi,2015-02-28 23:00,1396.669810,2.334365,56.828019,57.542000,112.396599,88.659606,1012.139679,0.051924,24.638602,13.628160,1.826647
4,Guanyuan,2015-02-28 23:00,1318.776455,3.685729,62.047019,53.958095,111.979798,86.233050,1010.738361,0.058728,23.514032,13.454356,1.521949
5,Gucheng,2015-02-28 23:00,1410.418128,2.718472,61.605072,55.328806,123.673331,86.702029,1008.446686,0.048319,19.957273,13.992583,1.099851
6,Huairou,2015-02-28 23:00,1063.971786,2.074332,37.428963,61.872431,97.840406,73.946730,1007.162974,0.057189,17.440170,12.455618,1.542807
7,Nongzhanguan,2015-02-28 23:00,1359.250000,2.334365,61.574250,56.022812,116.504311,88.955778,1012.139679,0.051924,25.077652,13.628160,1.826647
8,Shunyi,2015-02-28 23:00,1225.431397,2.454990,44.413416,55.669914,104.205190,82.429081,1012.457439,0.053237,17.284512,13.436608,1.769419
9,Tiantan,2015-02-28 23:00,1345.616170,2.334365,55.106711,54.031888,112.168025,86.203058,1012.139679,0.051924,19.535422,13.628160,1.826647


## Interpretation and limitations
Review the displayed evidence before drawing conclusions. Synthetic checks establish implementation behavior, not predictive performance. Training diagnostics cannot justify silently changing frozen settings. Audit missingness is structural evidence. A failed push stops execution with a local checkpoint retained; restart with the same revision/run ID to restore remotely confirmed work. In-flight work not yet uploaded can be lost on runtime termination. See `docs/P4_WORKFLOW.md` for commands and validation limits.